## Business Problem

Ref: [Link](https://www.youtube.com/watch?v=iCj4lT5KvJk)

As a marketing agency, our primary objective is to maximize the return on investment (ROI) for our clients' advertising campaigns. We have conducted two ad campaigns, one on Facebook and the other on AdWords, and we need to determine which platform yields better results in terms of clicks, conversions, and overall cost-effectiveness. By identifying the most effective platform, we can allocate our resources more efficiently and optimize our advertising strategies to deliver better outcomes for our clients.

---

## Research Question

Which ad platform is more effective in terms of conversions, clicks, and overall cost-effectiveness?

---

## Importing Libraries


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as st

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import coint

import warnings
warnings.filterwarnings('ignore')


## Data Description

The dataset comprises a collection of data comparing the performance of two separate ad campaigns conducted throughout the year 2019. Specifically, the data covers a **Facebook Ad campaign** and an **AdWords Ad campaign**. For each day of the year 2019, there is a corresponding row in the dataset, resulting in a total of **365 lines of campaign data to analyze**. The dataset includes various performance metrics for each ad campaign, providing insights into their effectiveness and efficiency over time.

**Key features included in the dataset are as follows:**

* **Date:** The date corresponding to each row of campaign data, ranging from **January 1st, 2019 to December 31st, 2019**.
* **Ad Views:** The number of times the ad was viewed.
* **Ad Clicks:** The number of clicks received on the ad.
* **Ad Conversions:** The number of conversions resulting from the ad.
* **Cost per Ad:** The cost associated with running the Facebook ad campaign.
* **Click-Through Rate (CTR):** The ratio of clicks to views, indicating the effectiveness of the ad in generating clicks.
* **Conversion Rate:** The ratio of conversions to clicks, reflecting the effectiveness of the ad in driving desired actions.
* **Cost per Click (CPC):** The average cost incurred per click on the ad.


In [0]:
# loading the dataset
df = pd.read_csv('marketing_campaign.csv')

In [0]:
# data overview
df.head()

In [0]:
# rows and columns count of the dataset
df.shape

In [0]:
# datatypes of the columns
df.dtypes

In [0]:
# converting date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# descriptive stats of the campaigns
df.describe()

## Comparing Campaigns performance

In [0]:
# distribution of the clicks and conversions
plt.figure(figsize=(15,6))
plt.subplot(1,2,1)
plt.title('Facebook Ad Clicks')
sns.histplot(df['Facebook Ad Clicks'], bins = 7, edgecolor = 'k', kde = True)

plt.subplot(1,2,2)
plt.title('Facebook Ad Conversions')
sns.histplot(df['Facebook Ad Conversions'], bins = 7, edgecolor = 'k', kde = True)

plt.show()


plt.figure(figsize=(15,6))
plt.subplot(1,2,1)
plt.title('Adwords Ad Clicks')
sns.histplot(df['Adwords Ad Clicks'], bins = 7, edgecolor = 'k', kde = True)

plt.subplot(1,2,2)
plt.title('Adwords Ad Conversions')
sns.histplot(df['Adwords Ad Conversions'], bins = 7, edgecolor = 'k', kde = True)

plt.show()

All the histogram are showing somewhat symmetrical shape. This symmetrical shape suggests that the number of clicks and conversions is relatively evenly distributed. In other words, there are not many clicks or conversions that are outliers on either the high or low end.

### How frequently do we observe days with high numbers of conversions compared to days with low numbers of conversions?

In [0]:
# creating function to calculate the category for the conversions
def create_conversion_category(conversion_col):
    category = []
    
    for conversion in df[conversion_col]:
        if conversion < 6:
            category.append('less than 6')
        elif 6 <= conversion < 11:
            category.append('6 - 10')
        elif 11 <= conversion < 16:
            category.append('10 - 15')
        else:
            category.append('more than 15')
            
    return category


# applying function of different campaign's conversions
df['Facebook Conversion Category'] = create_conversion_category('Facebook Ad Conversions')
df['Adwords Conversion Category'] = create_conversion_category('Adwords Ad Conversions')

In [0]:
df[['Facebook Ad Conversions',
    'Facebook Conversion Category',
    'Adwords Ad Conversions',
    'Adwords Conversion Category']].head()

In [0]:
df['Facebook Conversion Category'].value_counts()

In [0]:
facebook = pd.DataFrame(df['Facebook Conversion Category'].value_counts()).reset_index().rename(
    columns={'Facebook Conversion Category':'Count'}
)

facebook

In [0]:
df['Adwords Conversion Category'].value_counts()

In [0]:
adwords = pd.DataFrame(df['Adwords Conversion Category'].value_counts()).reset_index().rename(
    columns={'Adwords Conversion Category': 'Category'}
)
adwords

In [0]:
category_df = pd.merge(facebook, adwords, on='Category', how='outer').fillna(0)
category_df

In [0]:
category_df = category_df.iloc[[3,1,0,2]]
category_df

In [0]:
X_axis = np.arange(len(category_df))

plt.figure(figsize = (15,6))

plt.bar(X_axis - 0.2, category_df['count_x'], 0.4,
        label = 'Facebook', color = '#03989E', linewidth = 1, edgecolor = 'k')

plt.bar(X_axis + 0.2, category_df['count_y'], 0.4,
        label = 'Adwords', color = '#A62372', linewidth = 1, edgecolor = 'k')

plt.xticks(X_axis, category_df['Category'])
plt.xlabel("Conversion Category")
plt.ylabel("Number of Days")
plt.title("Number of Days by Conversion Category for Facebook and Adwords")
plt.legend()

plt.show()

- The data suggests Facebook had more frequent higher conversion days than AdWords, which either had very low conversion [cut off] (6 - 10).
- There is a significant variance in the number of high-conversion days between two different campaigns.
- The absence of any days with conversions between 10 - 15 and more than 15 in AdWords indicates a need to review what strategies were changed or what external factors could have influenced these numbers.

### Do more clicks on the ad really lead to more sales?


In [0]:
# Plotting Scatter Plots for Facebook and AdWords
plt.figure(figsize=(15,6))
plt.subplot(1,2,1)
plt.title('Facebook')
sns.scatterplot(x = df['Facebook Ad Clicks'], y = df['Facebook Ad Conversions'], color = '#03989E')
plt.xlabel('Clicks')
plt.ylabel('Conversions')

plt.subplot(1,2,2)
plt.title('AdWords')
sns.scatterplot(x = df['AdWords Ad Clicks'], y = df['AdWords Ad Conversions'], color = '#03989E')
plt.xlabel('Clicks')
plt.ylabel('Conversions')
plt.show()

In [0]:
# Calculating Correlation for Facebook
facebook_corr = df[['Facebook Ad Conversions', 'Facebook Ad Clicks']].corr()
facebook_corr

In [0]:
# Calculating Correlation for AdWords
adwords_corr = df[['AdWords Ad Conversions', 'AdWords Ad Clicks']].corr()
adwords_corr

In [0]:
# Printing the Correlation Coefficients
print('Correlation Coeff \n-----------------')
print('Facebook :', round(facebook_corr.values[0,1], 2))
print('AdWords  :', round(adwords_corr.values[0,1], 2))

- A correlation coefficient of 0.87...
- This strong ...


## Hypothesis testing

Hypothesis: Advertising on Facebook will result in greater number of conversions compared to advertising on AdWords.

Null Hypothesis: Advertising on Facebook will not result in greater number of conversions compared to advertising on AdWords.

H0 : $\mu_{Facebook} \le \mu_{AdWords}$

Alternative Hypothesis: Advertising on Facebook will result in greater number of conversions compared to advertising on AdWords.

H1 : $\mu_{Facebook} > \mu_{AdWords}$



In [0]:
# We will use a one-tailed t-test to test the hypothesis.
### One-tailed t-test
# Calculating the mean of Facebook and AdWords
fb_mean = round(df['Facebook Ad Conversions'].mean(),2)
aw_mean = round(df['AdWords Ad Conversions'].mean(),2)

print('Facebook Mean :', fb_mean)
print('AdWords Mean  :', aw_mean)

# also this is a one side test, so we will use the right side of the t-distribution
t_stats, p_value = st.ttest_ind(df['Facebook Ad Conversions'], df['AdWords Ad Conversions'], equal_var= False)
print('t-stats :', round(t_stats,2))
print('p-value :', round(p_value,2))

if p_value < 0.05:
  print('Reject Null Hypothesis')
else:
  print('Accept Null Hypothesis')

- The mean number of conversions
- The T statistic (32.88) is a measure of


## Regression Analysis

### What will happen when I do go with the Facebook Ad? How many facebook ad conversions can I expect given a certain number of facebook ad clicks?

In [0]:
# independant variable
X = df[['Facebook Ad Clicks']]

# dependent variable
y = df['Facebook Ad Conversions']

# initializing and fitting Linear REgression model
reg_model = LinearRegression()
reg_model.fit(X, y)
prediction = reg_model.predict(X)

# model evaluation
r2 = r2_score(y, prediction) * 100
mse = mean_squared_error(y, prediction)
print('R2 Score :', round(r2,2))
print('MSE      :', round(mse,2))

In [0]:
plt.figure(figsize = (8, 6))
sns.scatterplot(x = X, y = y, color = '#03989E', label = 'Actual data points')
plt.plot(X, prediction, color = '#A62372', label = 'Best Fit Line')
plt.xlabel('Facebook Ad Clicks')
plt.ylabel('Facebook Ad Conversions')
plt.title('Facebook Ad Clicks vs Facebook Ad Conversions')
plt.show()

In [0]:
print(f'For {50} Clicks, Expected Conversion: {round(reg_model.predict([[50]])[0],2)}')
print(f'For {80} Clicks, Expected Conversion: {round(reg_model.predict([[80]])[0],2)}')

- The model has
- With the insights
- For instance

## Analysing Facebook Campaign metrics over time

### Is there a long term equilibrium relationship between advertising spend and conversion rates

In [0]:
# cointegretaion test